# 横磁場イジングモデル：今回扱う模型

このフォルダで扱う模型の定義と、記号・前提をまとめる。横磁場イジングモデルの一般的な説明は [README.md](README.md) を参照。

## 模型の定義

ある一つの方向のイジング相互作用に対して、それと垂直な向きの磁場を組み合わせたもの。

今回は以下のモデルを考える。

\begin{equation}
H = -J\sum_i{Z_i Z_{i+1}} -h\sum_i{X_i}
\end{equation}

- **第1項について**

    この項は隣り合うスピンを揃えようとする相互作用を表しています。

    実際に計算すると
    
    \begin{equation}
    \begin{aligned}
    Z_1 Z_0\ket{00} &= +1\ket{00} \\
    Z_1 Z_0\ket{11} &= +1\ket{11} \\
    Z_1 Z_0\ket{01} &= -1\ket{01} \\
    Z_1 Z_0\ket{10} &= -1\ket{10} \\
    \end{aligned}
    \end{equation}

    これから隣同士のスピンが揃っているとエネルギーが小さくなり、揃っていないとエネルギーが大きくなる。すなわちスピンが揃っている方が好まれるモデルになっている。

- **第2項について**

    スピンをZの軸からXの軸へ傾けるような量子揺らぎを表しています。

## 前提と記号

- サイト数を$n$とし、$i = 0, 1, \dots, n-1$と番号をつける。
- 特に断らない限り**開放境界条件**（$Z_{n-1}Z_0$の項を含めない）とする。周期境界条件の場合は明記する。
- $J>0$（隣り合うスピンが揃う強磁性）、$h\ge0$とする。
- 記号や量子ビットの並び順の規約は [docs/conventions.md](../../docs/conventions.md) を参照。

## 対称性と2つの極限

- **パリティ対称性**：$P = \prod_j X_j$は$H$と交換する（$[H, P] = 0$）。$P$は全スピンを$Z\to-Z$と反転させる操作で、イジング項$Z_jZ_{j+1}$も横磁場$X_j$もこの反転で変わらないためである。
- **$h=0$（横磁場なし）**：基底状態は$\ket{0\cdots0}$と$\ket{1\cdots1}$の2重縮退で、スピンが$z$方向に揃った**秩序相**（強磁性）になる。
- **$J=0$（相互作用なし）**：基底状態は全スピンが$x$方向を向いた$\ket{+\cdots+}$で、$z$方向の秩序がない**無秩序相**（常磁性）になる。

この2つの極限の間、$h = J$で量子相転移が起こる（[13_phase_transition.ipynb](13_phase_transition.ipynb)）。

以下のコードで、$n=2$のハミルトニアンの行列と、パリティ対称性$[H,P]=0$を確かめる。

In [1]:
import numpy as np
from functools import reduce

from qmb.operators import X
from qmb.models import tfim_hamiltonian
# 実装を見たいときは、Jupyterで関数名の後に ?? を付けて実行する（例: hamiltonian??）

# n=2, J=1, h=0.5 のハミルトニアン（01_exact_diag.ipynb の手計算の行列）
H2 = tfim_hamiltonian(2, J=1.0, h=0.5)
print(H2.real)

# パリティ P = Π X_j と H が交換することを確認（n=6）
n = 6
H = tfim_hamiltonian(n, J=1.0, h=0.7)
P = reduce(np.kron, [X] * n)
print("||[H, P]|| =", np.linalg.norm(H @ P - P @ H))

[[-1.  -0.5 -0.5  0. ]
 [-0.5  1.   0.  -0.5]
 [-0.5  0.   1.  -0.5]
 [ 0.  -0.5 -0.5 -1. ]]
||[H, P]|| = 0.0


## このフォルダのノートブック

**解き方**（各手法をこの模型に適用したもの。一般論は [methods/](../../methods/) と同じ番号）

| ノートブック | 内容 |
|---|---|
| [01_exact_diag.ipynb](01_exact_diag.ipynb) | 厳密対角化：$n=2$ の行列と手計算 |
| [02_jordan_wigner.ipynb](02_jordan_wigner.ipynb) | Jordan-Wigner変換：行列 $A, B$ の導出、境界条件、$n=2$ の手計算 |
| [03_trotter.ipynb](03_trotter.ipynb) | Trotter分解：ハミルトニアンの分け方と回路、誤差の特徴、$n=2$ の手計算 |
| [04_vqe.ipynb](04_vqe.ipynb) | VQE：アンザッツ（HVA）、測定、$n=2$ の手計算 |

**物理の問い**（複数の手法の比較）

| ノートブック | 調べること | 使う手法 |
|---|---|---|
| [11_ground_state.ipynb](11_ground_state.ipynb) | 基底エネルギー$E_0$ | 厳密対角化・Jordan-Wigner変換・VQE |
| [12_quench_dynamics.ipynb](12_quench_dynamics.ipynb) | 初期状態$\ket{+\cdots+}$からのクエンチ後の横磁化$m_x(t)$ | 厳密対角化・Jordan-Wigner変換・Trotter分解 |
| [13_phase_transition.ipynb](13_phase_transition.ipynb) | ギャップ、秩序変数、相転移点$h_c=J$ | （準備中） |
| [14_entanglement.ipynb](14_entanglement.ipynb) | もつれエントロピー、面積則 | （準備中） |

横磁化は
\begin{equation}
M_x = \frac{1}{n}\sum_{j}X_j,\quad m_x(t) = \bra{\psi(t)}M_x\ket{\psi(t)}
\end{equation}
で定義する。$\ket{+\cdots+}$は$J=0$の基底状態なので、「横磁場だけの系にイジング相互作用を急に加えたとき（クエンチ）にどう時間発展するか」を見ることになる。